# AuraSight — RF-DETR 痘痘检测训练

**模型：** RF-DETR（Roboflow Detection Transformer）— Roboflow 2025年发布的 Transformer 检测器，对小目标（痘痘）检测效果比 YOLOv8 更好。

**流程：**
1. 安装依赖
2. 从 Roboflow 下载数据集
3. 训练 RF-DETR
4. 评估结果
5. 导出权重

> ⚠️ **必须先切换 GPU Runtime！**
> Runtime → Change runtime type → **T4 GPU** → Save

## 0. 检查 GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('❌ 没有 GPU！请先切换到 GPU Runtime 再继续。')

## 1. 安装依赖

In [ ]:
!pip install rfdetr roboflow supervision --quiet

import rfdetr
print('rfdetr version:', rfdetr.__version__)

## 2. 从 Roboflow 下载数据集

**怎么找 Workspace / Project Slug？**
打开你的 Roboflow 项目，URL 格式：
```
https://app.roboflow.com/<workspace>/<project>/<version>
```
把下面三个变量替换成你自己的值。

In [ ]:
RF_API_KEY   = "ApbnDOP6wMTly0rp8Mmi"   # ← 你的 API Key
RF_WORKSPACE = "your-workspace-slug"     # ← 从 URL 里复制
RF_PROJECT   = "acne-detection-v1"       # ← 项目 slug
RF_VERSION   = 1                         # ← 版本号

from roboflow import Roboflow
rf = Roboflow(api_key=RF_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)

# RF-DETR 使用 COCO JSON 格式
dataset = project.version(RF_VERSION).download("coco")

print("✅ 数据集下载完成")
print("路径:", dataset.location)

In [ ]:
import os, glob

# 确认数据集结构
train_dir = os.path.join(dataset.location, "train")
valid_dir = os.path.join(dataset.location, "valid")

train_imgs = glob.glob(train_dir + "/images/*.jpg") + glob.glob(train_dir + "/*.jpg")
valid_imgs = glob.glob(valid_dir + "/images/*.jpg") + glob.glob(valid_dir + "/*.jpg")

print(f"训练集: {len(train_imgs)} 张图")
print(f"验证集: {len(valid_imgs)} 张图")

## 3. 训练 RF-DETR

| 参数 | 说明 | 建议值 |
|------|------|--------|
| `model_type` | 模型大小 | `rfdetr-base`（推荐）/ `rfdetr-large`（更准但慢） |
| `epochs` | 训练轮数 | 50–100 |
| `batch_size` | 批大小 | 4（T4 显存限制） |
| `imgsz` | 图片尺寸 | 560 |
| `lr` | 学习率 | 1e-4 |

> 1419 张图 + 50 epochs + rfdetr-base 在 T4 上大约 **30–50 分钟**。

In [ ]:
from rfdetr import RFDETRBase

model = RFDETRBase()  # 用 rfdetr-large 替换可获得更高精度，但需要更多显存

model.train(
    dataset_dir=dataset.location,
    epochs=80,
    batch_size=4,
    imgsz=560,
    lr=1e-4,
    lr_encoder=1e-5,      # encoder 用更小的学习率
    output_dir="./aurasight_rfdetr",
    # 数据增强
    gradient_checkpointing=True,  # 节省显存
)

print("\n✅ 训练完成！")

## 4. 评估结果

In [ ]:
from IPython.display import Image, display
import os

# 训练曲线
for plot_file in ["loss.png", "map.png", "results.png"]:
    path = f"./aurasight_rfdetr/{plot_file}"
    if os.path.exists(path):
        print(f"📊 {plot_file}")
        display(Image(path))

In [ ]:
import supervision as sv
import glob, random
import numpy as np
from PIL import Image as PILImage

# 找最好的权重
weight_files = glob.glob("./aurasight_rfdetr/*.pth") + glob.glob("./aurasight_rfdetr/checkpoint*.pth")
best_ckpt = sorted(weight_files)[-1] if weight_files else None
print("使用权重:", best_ckpt)

# 加载模型做预测
infer_model = RFDETRBase(pretrain_weights=best_ckpt)

# 随机抽 4 张验证图做可视化
val_imgs = glob.glob(dataset.location + "/valid/**/*.jpg", recursive=True)
sample = random.sample(val_imgs, min(4, len(val_imgs)))

for img_path in sample:
    img = PILImage.open(img_path)
    detections = infer_model.predict(img, threshold=0.35)
    
    annotator = sv.BoxAnnotator()
    annotated = annotator.annotate(np.array(img), detections)
    display(PILImage.fromarray(annotated).resize((400, 400)))
    print(f"  检测到 {len(detections)} 个痘痘")

## 5. 导出权重

In [ ]:
# 方式 A：下载 .pth 权重文件
from google.colab import files

if best_ckpt:
    files.download(best_ckpt)
    print(f"✅ 已下载: {best_ckpt}")
else:
    print("❌ 找不到权重文件，检查 aurasight_rfdetr 目录")

In [ ]:
# 方式 B：上传回 Roboflow（会创建新版本，可继续用 Hosted API）
# 注意：RF-DETR 上传到 Roboflow 需要先导出成指定格式
# 目前建议用方式 A 下载本地后手动部署

print("推荐流程：")
print("1. 下载 best .pth 文件")
print("2. 放到 AuraSight-api 目录")
print("3. 在后端用 rfdetr 包直接推理（告诉我，我帮你写 Node.js 调用代码）")

## 6. 后端集成方案

RF-DETR 训练出来的权重是 `.pth` 格式。集成到 `AuraSight-api` 有两种方式：

### 方案 A — 上传回 Roboflow Hosted API（推荐，省事）
在 Roboflow 平台直接用 RF-DETR 训练（消耗 credits），训练完后 Hosted API 直接可用，`.env` 里只需改 version 号：
```
ROBOFLOW_MODEL=acne-detection-v1-kf14t/2
```

### 方案 B — Python 微服务（不消耗 Roboflow credits）
写一个 FastAPI 服务加载 `.pth` 权重，Node.js 后端调这个 Python 服务。可以部署到 Render / Railway 免费层。告诉我要这个，我帮你写全套代码。

### 方案 C — 导出 ONNX 在 Node.js 内推理
RF-DETR 支持 ONNX 导出，用 `onnxruntime-node` 直接在 Node.js 里跑。性能好，不依赖第三方服务。